<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://www.sap.com/dam/application/shared/logos/sap-logo-svg.svg" alt="SAP" width="100">
</div>

# Exercise 1B: JIT Model Evaluation and Benchmarking
## Regression and Intervention-Risk Classification

Exercise 1A focused on using SAP-RPT-1 inside a JIT risk workflow. Exercise 1B adds the evaluation lens: how well does that workflow perform, how should we measure it, and what do we learn by comparing SAP-RPT-1 with a small set of open-source baselines on the same business problem?

### Learning Objectives

By the end of Exercise 1B, you will be able to:
- Compare two modeling formulations for the same JIT supply chain problem: delay-day regression and intervention-risk classification
- Evaluate workshop-grade model performance using metrics that support planner and procurement decisions
- Benchmark SAP-RPT-1 against a focused set of open-source ML baselines
- Explain when severity prediction is more useful than threshold-based intervention routing

### Evaluation Lens

This notebook keeps the same JIT use case from Exercise 1A and evaluates it through two decision frames:

1. **Regression:** How many days late is this PO likely to be?
2. **Classification:** Is this PO likely to cross the Red-risk threshold and require intervention?

Those questions support different business decisions. Regression is better for sizing severity and potential business impact. Classification is better for alert routing, planner attention, and escalation workflows.


The benchmark is workshop-grade. It is designed to compare approaches in a disciplined way, not to turn the exercise into generic ML experimentation.

In [11]:
%pip install generative-ai-hub-sdk==4.12.4 --quiet
%pip install pandas python-dotenv requests scikit-learn --quiet

print("Packages installed successfully. Restart the kernel if prompted.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Packages installed successfully. Restart the kernel if prompted.


In [1]:
import math
import os
import time
from typing import Any, Dict, List, Tuple

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

load_dotenv(find_dotenv(), override=False)

RISK_THRESHOLD_AMBER = 1.0
RISK_THRESHOLD_RED = 3.0
TEST_SIZE = 0.20
RANDOM_STATE = 42
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 60
TOKEN_REFRESH_BUFFER_SECONDS = 60

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
RPT1_DEPLOYMENT_URL = os.getenv("RPT1_DEPLOYMENT_URL")

USE_MOCK_MODE = not all([
    AICORE_AUTH_URL,
    AICORE_CLIENT_ID,
    AICORE_CLIENT_SECRET,
    AICORE_RESOURCE_GROUP,
    RPT1_DEPLOYMENT_URL,
])

print(f"Mock mode: {USE_MOCK_MODE}")

Mock mode: False


### Step 1 - Load the Exercise 1B Datasets

These files are isolated from Exercise 1A so the original workshop remains unchanged.

In [2]:
historical_df = pd.read_csv("data/historical_po_data_part1b.csv")
prediction_df = pd.read_csv("data/new_po_prediction_part1b.csv")

def derive_risk_tier(delay_days: float) -> str:
    if delay_days < RISK_THRESHOLD_AMBER:
        return "Green"
    if delay_days <= RISK_THRESHOLD_RED:
        return "Amber"
    return "Red"

historical_df["Risk_Tier"] = historical_df["Actual_Delay_Days"].apply(derive_risk_tier)
historical_df["Needs_Intervention"] = historical_df["Actual_Delay_Days"] > RISK_THRESHOLD_RED

print(f"Historical rows: {len(historical_df):,}")
print(f"Prediction scenarios: {len(prediction_df):,}")
print("Risk tier distribution:")
display(historical_df["Risk_Tier"].value_counts().rename_axis("Risk_Tier").to_frame("Count"))

Historical rows: 624
Prediction scenarios: 4
Risk tier distribution:


,Count
Risk_Tier,
Amber,322
Green,201
Red,101


### Step 2 - Prepare Features

We keep the same structured ERP-style features used in Exercise 1A. For workshop purposes, we use the provided fields as-is. In a production benchmark, derived supplier features would need stricter train-only recomputation controls.

In [3]:
feature_columns = [
    "Vendor_ID", "Vendor_Country", "Vendor_OTIF_Percent", "Vendor_Avg_Past_Delay",
    "Material_ID", "Material_Group", "Criticality_Flag", "Plant_ID",
    "Order_Quantity", "Net_Price", "Planned_Lead_Time_Days", "Order_Month", "Incoterms"
]
categorical_features = [
    "Vendor_ID", "Vendor_Country", "Material_ID", "Material_Group",
    "Criticality_Flag", "Plant_ID", "Incoterms"
]
numeric_features = [column for column in feature_columns if column not in categorical_features]

X = historical_df[feature_columns].copy()
y_reg = historical_df["Actual_Delay_Days"].copy()
y_cls = historical_df["Needs_Intervention"].astype(int).copy()

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_cls
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
    ]
)

print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Positive intervention cases in test split: {int(y_cls_test.sum())}")

Train rows: 499 | Test rows: 125
Positive intervention cases in test split: 20


### Step 3 - Open-Source Regression Baselines

We start with two simple baselines: a linear model and a tree-based ensemble.

In [4]:
def evaluate_regression(y_true: pd.Series, y_pred: List[float]) -> Dict[str, float]:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

regression_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    ),
}

regression_results = []
fitted_regressors = {}

for model_name, model in regression_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_reg_train)
    predictions = pipeline.predict(X_test)
    metrics = evaluate_regression(y_reg_test, predictions)
    metrics["Model"] = model_name
    regression_results.append(metrics)
    fitted_regressors[model_name] = pipeline

regression_results_df = pd.DataFrame(regression_results).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
display(regression_results_df)

,MAE,RMSE,R2,Model
0,0.400620,0.498120,0.799684,Random Forest Regressor
1,0.407981,0.503111,0.795649,Linear Regression


### Step 4 - SAP-RPT-1 Holdout Evaluation

This section uses the same SAP-RPT-1 call pattern as Exercise 1A. If AI Core credentials are not available, the notebook falls back to the Exercise 1A mock proxy so the workflow still runs.

The table below compares SAP-RPT-1 with the open-source regression baselines using three metrics:

- **MAE (Mean Absolute Error):** On average, how many days off was the prediction?
  Lower is better. A MAE of `0.32` means the model is off by about `0.32` days on average.
- **RMSE (Root Mean Squared Error):** Similar to MAE, but it penalizes larger misses more heavily.
  Lower is better. Use this to see whether a model sometimes makes more severe mistakes, even if the average looks acceptable.
- **R2:** How much of the variation in actual delay the model explains.
  Higher is better. `1.0` is perfect, `0.0` means the model is no better than a very simple average-based baseline, and negative values mean the model is performing poorly.

For business interpretation:
- If you care about **average planning accuracy**, look first at MAE.
- If you care about **avoiding large misses on high-risk orders**, pay close attention to RMSE.
- If you want a quick sense of **overall explanatory strength**, use R2 as a supporting metric.

In [5]:
class AICoreClient:
    def __init__(self, auth_url: str, client_id: str, client_secret: str, base_url: str, resource_group: str):
        self._auth_url = auth_url
        self._client_id = client_id
        self._client_secret = client_secret
        self._base_url = base_url
        self._resource_group = resource_group
        self._access_token: str | None = None
        self._token_expires_at: float = 0.0

    def _get_token(self) -> str:
        now = time.time()
        if self._access_token and now < (self._token_expires_at - TOKEN_REFRESH_BUFFER_SECONDS):
            return self._access_token

        response = requests.post(
            f"{self._auth_url}/oauth/token",
            data={"grant_type": "client_credentials"},
            auth=(self._client_id, self._client_secret),
            timeout=TOKEN_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json()
        self._access_token = payload["access_token"]
        self._token_expires_at = now + int(payload.get("expires_in", 600))
        return self._access_token

    def predict(self, deployment_url: str, payload: dict) -> dict:
        token = self._get_token()
        response = requests.post(
            deployment_url,
            json=payload,
            headers={
                "Authorization": f"Bearer {token}",
                "AI-Resource-Group": self._resource_group,
                "Content-Type": "application/json",
            },
            timeout=HTTP_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        return response.json()

def mock_rpt1_predict(row: pd.Series) -> float:
    vendor = row["Vendor_ID"]
    month = int(row["Order_Month"])
    qty = int(row["Order_Quantity"])
    material_group = row["Material_Group"]
    incoterms = row["Incoterms"]

    delay = 0.0
    if vendor == "VENDOR_A":
        delay += 0.5
    elif vendor == "VENDOR_B":
        delay += 1.8
    else:
        delay += 2.6

    if vendor == "VENDOR_B" and month in [10, 11, 12]:
        delay += 1.5
    if vendor == "VENDOR_C" and qty > 1500:
        delay += 1.2
    if material_group in ["Control Chip", "Optical Module"]:
        delay += 0.3
    if incoterms == "FOB":
        delay += 0.2
    elif incoterms == "DAP":
        delay -= 0.2

    return round(max(delay, 0), 1)

aicore_client = None if USE_MOCK_MODE else AICoreClient(
    auth_url=AICORE_AUTH_URL,
    client_id=AICORE_CLIENT_ID,
    client_secret=AICORE_CLIENT_SECRET,
    base_url=AICORE_BASE_URL,
    resource_group=AICORE_RESOURCE_GROUP,
)

def predict_delay_rpt1(row: pd.Series, context_df: pd.DataFrame) -> Tuple[float, str]:
    if USE_MOCK_MODE or aicore_client is None:
        return mock_rpt1_predict(row), "mock-proxy"

    columns = feature_columns + ["Actual_Delay_Days"]
    context_sample = context_df[columns].sample(n=min(200, len(context_df)), random_state=RANDOM_STATE)
    pred_row = row[feature_columns].to_dict()
    pred_row["Actual_Delay_Days"] = "[PREDICT]"
    rows = pd.concat([context_sample, pd.DataFrame([pred_row])], ignore_index=True).to_dict("records")
    payload = {
        "rows": rows,
        "prediction_config": {
            "target_columns": [{
                "name": "Actual_Delay_Days",
                "prediction_placeholder": "[PREDICT]"
            }]
        }
    }

    deployment_url = RPT1_DEPLOYMENT_URL.rstrip("/")
    if not deployment_url.endswith("/predict"):
        deployment_url = f"{deployment_url}/predict"

    try:
        response = aicore_client.predict(deployment_url=deployment_url, payload=payload)
        prediction_value = response["predictions"][0]["Actual_Delay_Days"][0]["prediction"]
        return float(prediction_value), "sap-rpt-1"
    except Exception as exc:
        print(f"[WARNING] SAP-RPT-1 evaluation fallback: {exc}")
        return mock_rpt1_predict(row), "mock-fallback"

rpt1_predictions = []
rpt1_methods = []
for _, row in X_test.reset_index(drop=True).iterrows():
    prediction, method = predict_delay_rpt1(row, historical_df)
    rpt1_predictions.append(prediction)
    rpt1_methods.append(method)

rpt1_metrics = evaluate_regression(y_reg_test.reset_index(drop=True), rpt1_predictions)
rpt1_metrics["Model"] = "SAP-RPT-1"
rpt1_metrics["Inference_Mode"] = pd.Series(rpt1_methods).mode().iloc[0]

regression_comparison_df = pd.concat([
    regression_results_df,
    pd.DataFrame([rpt1_metrics])
], ignore_index=True, sort=False).sort_values(["MAE", "RMSE"]).reset_index(drop=True)

display(regression_comparison_df)

,MAE,RMSE,R2,Model,Inference_Mode
0,0.318652,0.462264,0.827484,SAP-RPT-1,sap-rpt-1
1,0.400620,0.498120,0.799684,Random Forest Regressor,NaN
2,0.407981,0.503111,0.795649,Linear Regression,NaN


### Step 5 - Intervention-Risk Classification Benchmark

For operational routing, planners often care less about the exact delay number and more about whether the PO crosses the Red-risk threshold.

The table below compares classification performance using three decision-oriented metrics:

- **Precision:** Of the POs the model flagged as needing intervention, how many actually did?
  Higher is better. High precision means fewer false alarms and less wasted planner attention.
- **Recall:** Of the POs that truly needed intervention, how many did the model catch?
  Higher is better. High recall means fewer dangerous misses.
- **F1:** A balance between Precision and Recall.
  Higher is better. Use F1 when you want one combined score instead of optimizing only for false alarms or only for missed risks.

For business interpretation:
- If the business wants to **avoid unnecessary escalations**, prioritize Precision.
- If the business wants to **catch as many true Red-risk cases as possible**, prioritize Recall.
- If the business wants a **balanced alerting model**, compare F1 first.

The `Confusion_Matrix` is included for transparency. It shows how many cases were classified correctly versus incorrectly, so you can see the trade-off between false alarms and missed interventions more directly.

In [6]:
def evaluate_classification(y_true: pd.Series, y_pred: List[int]) -> Dict[str, float]:
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear"
    ),
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        class_weight="balanced"
    ),
}

classification_results = []
fitted_classifiers = {}

for model_name, model in classification_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_cls_train)
    predictions = pipeline.predict(X_test)
    metrics = evaluate_classification(y_cls_test, predictions)
    metrics["Model"] = model_name
    metrics["Confusion_Matrix"] = str(confusion_matrix(y_cls_test, predictions).tolist())
    classification_results.append(metrics)
    fitted_classifiers[model_name] = pipeline

rpt1_class_predictions = [int(prediction > RISK_THRESHOLD_RED) for prediction in rpt1_predictions]
rpt1_cls_metrics = evaluate_classification(y_cls_test, rpt1_class_predictions)
rpt1_cls_metrics["Model"] = "SAP-RPT-1 Thresholded"
rpt1_cls_metrics["Confusion_Matrix"] = str(confusion_matrix(y_cls_test, rpt1_class_predictions).tolist())

classification_results_df = pd.concat([
    pd.DataFrame(classification_results),
    pd.DataFrame([rpt1_cls_metrics])
], ignore_index=True).sort_values(["F1", "Recall"], ascending=False).reset_index(drop=True)

display(classification_results_df)

,Precision,Recall,F1,Model,Confusion_Matrix
0,0.764706,0.65,0.702703,SAP-RPT-1 Thresholded,"[[101, 4], [7, 13]]"
1,0.515152,0.85,0.641509,Random Forest Classifier,"[[89, 16], [3, 17]]"
2,0.450000,0.90,0.600000,Logistic Regression,"[[83, 22], [2, 18]]"


### Step 6 - Score New Scenarios

This final step applies the strongest open-source baseline and the SAP-RPT-1 path to a few new POs so you can compare operational outputs side by side.

**How to read this table:**
- This is a **scenario comparison**, not the primary benchmark for deciding which model is best overall.
- Use **Step 4** for the formal regression comparison and **Step 5** for the formal classification comparison.
- In Step 6, focus on whether the models lead to the **same or different business actions** for each PO, even if the exact predicted delay values are not identical.
- If both models assign the same risk tier, that usually means they are directionally aligned from an operational point of view.

In [7]:
best_regressor_name = regression_results_df.sort_values(["MAE", "RMSE"]).iloc[0]["Model"]
best_classifier_name = classification_results_df.sort_values(["F1", "Recall"], ascending=False).iloc[0]["Model"]

best_regressor = fitted_regressors[best_regressor_name]
best_classifier = fitted_classifiers.get(best_classifier_name)
if best_classifier is None:
    best_classifier_name = "Random Forest Classifier"
    best_classifier = fitted_classifiers[best_classifier_name]

scenario_rows = prediction_df[feature_columns].copy()
prediction_df["OpenSource_Predicted_Delay"] = best_regressor.predict(scenario_rows).round(2)
prediction_df["OpenSource_Risk_Tier"] = prediction_df["OpenSource_Predicted_Delay"].apply(derive_risk_tier)
prediction_df["OpenSource_Intervention_Flag"] = best_classifier.predict(scenario_rows).astype(int)

rpt1_scores = []
rpt1_modes = []
for _, row in prediction_df.iterrows():
    score, mode = predict_delay_rpt1(row, historical_df)
    rpt1_scores.append(round(score, 2))
    rpt1_modes.append(mode)

prediction_df["SAP_RPT1_Predicted_Delay"] = rpt1_scores
prediction_df["SAP_RPT1_Risk_Tier"] = prediction_df["SAP_RPT1_Predicted_Delay"].apply(derive_risk_tier)
prediction_df["SAP_RPT1_Mode"] = rpt1_modes

summary_columns = [
    "PO_ID", "Vendor_ID", "Material_Group", "Criticality_Flag",
    "OpenSource_Predicted_Delay", "OpenSource_Risk_Tier", "OpenSource_Intervention_Flag",
    "SAP_RPT1_Predicted_Delay", "SAP_RPT1_Risk_Tier", "SAP_RPT1_Mode"
]

print(f"Best regression baseline: {best_regressor_name}")
print(f"Best classification baseline: {best_classifier_name}")
display(prediction_df[summary_columns])

Best regression baseline: Random Forest Regressor
Best classification baseline: Random Forest Classifier


,PO_ID,Vendor_ID,Material_Group,Criticality_Flag,OpenSource_Predicted_Delay,OpenSource_Risk_Tier,OpenSource_Intervention_Flag,SAP_RPT1_Predicted_Delay,SAP_RPT1_Risk_Tier,SAP_RPT1_Mode
0,PO210001,VENDOR_A,Sensor,Yes,0.51,Green,0,0.69,Green,sap-rpt-1
1,PO210002,VENDOR_B,Optical Module,Yes,3.86,Red,1,3.54,Red,sap-rpt-1
2,PO210003,VENDOR_C,Control Chip,Yes,4.55,Red,1,3.54,Red,sap-rpt-1
3,PO210004,VENDOR_B,Battery Unit,No,1.58,Amber,0,2.09,Amber,sap-rpt-1


## Summary

Exercise 1B extends the same JIT use case with an evaluation and benchmarking lens:
- **Regression** helps estimate delay severity and potential business impact
- **Classification** helps route intervention decisions faster
- **SAP-RPT-1 versus open-source baselines** gives a practical comparison point for architecture and delivery choices

Use the results in this notebook to discuss what is strong enough for advisory rollout, what business decisions each modeling approach supports best, and what additional evidence would be needed before broader operationalization.